Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Homework 2b: photo-z agreement sky map

**Task:** plot the fraction of objects with matching Legacy Survey and DP2 photo-z as a sky map.

**Plan:**
1. Open the catalog saved in `Notebook_2a_Crossmatch`
2. Compute the fraction of matching photo-z in each HEALPix pixel
3. Plot it as a sky map

If you'd like to solve it yourself, `Notebook_2a_Crossmatch` may help.

> [!NOTE]
> This notebook uses only the shareable catalog saved in `Notebook_2a_Crossmatch`, so anyone who has received this catalog can run it, including non-DRHs.

## Solution

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import astropy.units as u
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from dask.distributed import Client
from hats.inspection.visualize_catalog import plot_healpix_map
from hats.pixel_math.spatial_index import spatial_index_to_healpix
from matplotlib.colors import Normalize

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)

## Setup Dask cluster

We use the same Dask setup as in `Notebook_1a_Intro`: sixteen workers, one per CPU of a "Large" RSP container, with one thread and 2 GB of memory each, which is enough for this low-memory pipeline, and `/deleted-sundays/` for spilling data to disk.

If you are not on RSP, set `local_directory` to a scratch directory on your platform.

In [ ]:
client = Client(
    n_workers=16,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

### Open the catalog

All of its columns are shareable, so we can look at the data.

In [ ]:
shared = lsdb.open_catalog("lsdr11_rubindp2_photoz_match")
shared.head()

### Fraction of matching photo-z per HEALPix pixel

The catalog index, `_healpix_29`, is the HEALPix pixel index at order 29. We convert it to a lower order and count objects in each pixel with `map_partitions()`, as we did for the histograms in `Notebook_1a_Intro`. A pixel may span several partitions, so we sum the partial counts at the end. Order 9 corresponds to pixels of about 6.9 arcmin. We skip pixels with too few objects.

In [ ]:
ORDER = 9
MIN_COUNT = 10


def per_pixel_counts(df):
    """Number of objects and number of matching photo-z per HEALPix pixel for a single partition."""
    pixel = spatial_index_to_healpix(df.index.to_numpy(), target_order=ORDER)
    counts = df["lsdr11_rubindp2_photoz_match"].groupby(pixel).agg(["size", "sum"])
    return pd.DataFrame(
        {"pixel": counts.index, "n": counts["size"].to_numpy(), "n_match": counts["sum"].to_numpy()}
    )


partial_counts = shared.map_partitions(per_pixel_counts, meta={"pixel": int, "n": int, "n_match": int})

In [ ]:
per_pixel = partial_counts.compute().groupby("pixel")[["n", "n_match"]].sum()
per_pixel = per_pixel[per_pixel["n"] >= MIN_COUNT]
per_pixel["fraction"] = per_pixel["n_match"] / per_pixel["n"]
print(f"Number of pixels: {len(per_pixel):,}")

### Sky map

We plot the fractions with `plot_healpix_map()`, see the ["Plotting aggregations of data"](https://docs.lsdb.io/en/latest/tutorials/pre_executed/plotting.html#5.-Plotting-aggregations-of-data) section of the LSDB plotting tutorial for more examples.

In [ ]:
fig, ax = plot_healpix_map(
    per_pixel["fraction"].to_numpy(),
    ipix=per_pixel.index.to_numpy(),
    depth=np.full(len(per_pixel), ORDER),
    fov=(12 * u.deg, 12 * u.deg),
    center=SkyCoord(300.0, -20.0, unit="deg"),
    norm=Normalize(vmin=0.0, vmax=1.0),
    title="Fraction of matching Legacy Survey and DP2 photo-z",
)
plt.show()

In [ ]:
client.close()